In [1]:
import json
import os
import requests
import time
import random
from dotenv import load_dotenv
from pathlib import Path
import yfinance as yf
import pandas as pd
import numpy as np
from edgar import *
from tqdm.notebook import tqdm

from util import *


load_dotenv(override=True)

ALPHA_VANTAGE_KEY = os.getenv('ALPHA_VANTAGE_KEY')
# IDENTITY = os.getenv("IDENTITY")
set_identity(IDENTITY)

CHECKPOINT_FILE = Path("checkpoint.json")

BASE_URL = "https://www.alphavantage.co/query"

# COMPANY_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
LISTING_PATH = 'us_stock_listing.csv'

# SEC_HEADERS = {
#     "User-Agent": f"financial-research-masters-degree ({IDENTITY})",
#     "Accept": "application/json"
# }

FUNCTIONS = {
    "INCOME_STATEMENT": "income_statement.json",
    "BALANCE_SHEET": "balance_sheet.json",
    "CASH_FLOW": "cash_flow.json",
}

IS_PREMIUM = True

RAW_DIR = Path("raw_data")
RAW_DIR.mkdir(exist_ok=True)

# PRICE_CACHE_DIR = Path("price_cache")
# PRICE_CACHE_DIR.mkdir(exist_ok=True)

In [9]:
ticker = yf.Ticker('DB')
hist = ticker.history(period='max')


In [10]:
hist.tail(3)

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-05-20 00:00:00-04:00,31.500000,32.880001,31.410000,32.869999,5100000,0.0,0.0
2026-05-21 00:00:00-04:00,32.330002,33.130001,32.220001,32.900002,3818900,0.0,0.0
2026-05-22 00:00:00-04:00,32.770000,32.810001,32.340000,32.430000,2048800,0.0,0.0


In [11]:
print(ticker.info.get('financialCurrency'), ticker.info.get('currency'))

EUR USD


In [2]:

def fetch(function: str, ticker: str):
    params = {
        "function": function,
        "symbol": ticker,
        "apikey": ALPHA_VANTAGE_KEY
    }

    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()

    data = r.json()

    if "Information" in data or "Note" in data:
        print(f"Error: {data.get("Information")}")
        raise RuntimeError("ALPHA_VANTAGE_LIMIT")
    
    if "quarterlyReports" not in data and "annualReports" not in data:
        raise RuntimeError("INVALID_ALPHA_RESPONSE")

    return data

def download_company(ticker: str):
    company_dir = RAW_DIR / ticker
    company_dir.mkdir(exist_ok=True)

    for func, filename in FUNCTIONS.items():

        filepath = company_dir / filename

        if filepath.exists():
            print("skip", ticker, filename)
            continue

        print("download", ticker, func)

        data = fetch(func, ticker)

        with open(filepath, 'w') as f:
            json.dump(data, f, indent=2)

        # Alpha Vantage rate limit, update constant based on key type
        if IS_PREMIUM:
            time.sleep(1)
        else:
            time.sleep(12)

In [20]:

def load_raw(ticker: str):
    base = RAW_DIR / ticker

    with open(base / "income_statement.json") as f:
        income = json.load(f)

    with open(base / "balance_sheet.json") as f:
        balance = json.load(f)

    with open(base / "cash_flow.json") as f:
        cash = json.load(f)

    return income, balance, cash

def merge_reports(income: dict, balance: dict, cash: dict, key: str) -> pd.DataFrame | None:
    '''
        key can be quarterlyReports | annualReports
    '''
    dfs = [
        pd.DataFrame(src.get(key, []))
        for src in (income, balance, cash)
    ]

    if any(df.empty for df in dfs):
        return None

    merged = dfs[0]
    for df in dfs[1:]:
        merged = merged.merge(df, on='fiscalDateEnding', how='outer', suffixes=("", "_dup"))
        merged.drop(columns=merged.filter(like="_dup").columns, inplace=True)
    
    merged["fiscalDateEnding"] = pd.to_datetime(merged["fiscalDateEnding"])

    merged = merged[merged["fiscalDateEnding"] >= FIRST_USABLE_DATE]

    NON_NUMERIC = {'fiscalDateEnding', 'reportedCurrency'}
    numeric_cols = [c for c in merged.columns if c not in NON_NUMERIC]
    merged[numeric_cols] = merged[numeric_cols].apply(
        lambda col: pd.to_numeric(col.str.replace(',', '', regex=False), errors='coerce')
    )

    merged.sort_values("fiscalDateEnding", inplace=True)
    merged.reset_index(drop=True, inplace=True)

    return merged


def build_reports(
        ticker: str,
        metadata: dict,
        period: str,
        price_history: pd.DataFrame = None,
        spy_price_history: pd.DataFrame = None,
    ) -> None:
    """
    period: 'quarterly' | 'yearly'
    """
    assert period in ('quarterly', 'yearly'), "period must be 'quarterly' or 'yearly'"

    report_key = 'quarterlyReports' if period == 'quarterly' else 'annualReports'
    output_dir = f"data_{period}"

    income, balance, cash = load_raw(ticker)

    merged = merge_reports(income, balance, cash, report_key)
    if merged is None:
        print(f'[{ticker}] has incomplete data for period {period}, skipping')
        return

    merged_final = get_data_after_company_public(merged, price_history)

    first, last = trim_to_continuous_range(merged_final['fiscalDateEnding'], period)
    if first != 0 or last != len(merged_final) - 1:
        print(
            f"WARNING [{ticker}] data cut to range "
            f"{merged_final.iloc[first]['fiscalDateEnding'].strftime('%Y-%m-%d')} - "
            f"{merged_final.iloc[last]['fiscalDateEnding'].strftime('%Y-%m-%d')} due to missing rows"
        )

    merged_final = merged_final.iloc[first:last + 1].reset_index(drop=True)

    currencies = merged_final['reportedCurrency'].unique().tolist()
    if len(currencies) > 2 and any(pd.isna(c) or c == 'None' for c in currencies):
        print(f"ERROR [{ticker}] Impossible to determine currency for some rows, 2 or more currencies and None values present, skipping")
        return

    merged_final = convert_values_to_usd(merged_final)
    if merged_final is None:
        print(f"ERROR [{ticker}] Converting to USD failed, skipping")
 
    # Stop if ticker has less than 5 Y / 17 Q of data (3 years back and one year into the future)
    min_required_rows = 17 if period == 'quarterly' else 5
    if len(merged_final) < min_required_rows:
        print(f"ERROR [{ticker}] only contains {len(merged_final)} data rows, min required is {min_required_rows}, skipping")
        return

    if len(merged) != len(merged_final):
        print(
            f"WARNING [{ticker}] Data cut due to unavailability of stock prices, "
            f"new first date: {merged_final.iloc[0]['fiscalDateEnding'].strftime('%Y-%m-%d')}"
        )

    if price_history is None:
        price_history = get_price_history(ticker)
    
    if spy_price_history is None:
        spy_price_history = get_price_history('SPY')

    report_dates = get_report_dates(ticker, period)

    default_form = '10-Q' if period == 'quarterly' else '10-K'
    default_fallback = FORM_FALLBACK_DAYS[default_form]

    def make_fallback_row():
        return {'filing_date': pd.NaT, 'reportDate': pd.NaT, 'form': default_form}

    if report_dates is None:
        print(f"WARNING: [{ticker}] No report dates found, using full default fallback for all rows")
        aligned_dates = pd.DataFrame([make_fallback_row() for _ in range(len(merged_final))])
    else:
        anchor_merged_idx = None

        for pos_m, (_, mrow) in enumerate(merged_final.iterrows()):
            fiscal_ts = pd.Timestamp(mrow['fiscalDateEnding'])
            for pos_d, (_, drow) in enumerate(report_dates.iterrows()):
                if abs(fiscal_ts - pd.Timestamp(drow['reportDate'])).days <= default_fallback:
                    anchor_merged_idx = pos_m
                    break
            if anchor_merged_idx is not None:
                break
        
        if anchor_merged_idx is None:
            print(f"WARNING: [{ticker}] No matching reportDate found for any fiscalDateEnding, using full default fallback ({default_fallback}d)")
            aligned_dates = pd.DataFrame([make_fallback_row() for _ in range(len(merged_final))])
        else:
            report_dates_ts = pd.to_datetime(report_dates['reportDate'])
            aligned_rows = []
            used_positions = set()

            for _, mrow in merged_final.iterrows():
                fiscal_ts = pd.Timestamp(mrow['fiscalDateEnding'])

                # find closest unused reportDate within tolerance
                best_pos = None
                best_delta = None
                for pos_d, report_ts in enumerate(report_dates_ts):
                    if pos_d in used_positions:
                        continue
                    delta = abs(fiscal_ts - report_ts).days
                    if delta <= default_fallback:
                        if best_delta is None or delta < best_delta:
                            best_pos = pos_d
                            best_delta = delta

                if best_pos is not None:
                    used_positions.add(best_pos)
                    aligned_rows.append(report_dates.iloc[best_pos].to_dict())
                else:
                    aligned_rows.append(make_fallback_row())

            aligned_dates = pd.DataFrame(aligned_rows).reset_index(drop=True)
            
            n_fallback = sum(1 for r in aligned_rows if pd.isna(r['filing_date']))
            if n_fallback > 0:
                print(f"WARNING: [{ticker}] {n_fallback} rows using default fallback (no matching reportDate)")
    
    df = pd.concat([
        merged_final.reset_index(drop=True),
        aligned_dates.reset_index(drop=True)
    ], axis=1)

    fallback_days = df['form'].map(FORM_FALLBACK_DAYS).fillna(45)
    delta = (df['fiscalDateEnding'] - pd.to_datetime(df['filing_date'])).dt.days.abs()

    has_filing = df['filing_date'].notna() & delta.notna() & (delta <= fallback_days)

    df['filing_date_used'] = np.where(
        has_filing,
        df['filing_date'].astype(str),
        (df['fiscalDateEnding'] + pd.to_timedelta(fallback_days, unit='D')).astype(str)
    )

    df['stock_price'] = lookup_prices_vectorized(price_history, pd.to_datetime(df['filing_date_used']))

    df = apply_accounting_logic_av(df)

    stock_diff_pct = count_future_diff_pct(df['stock_price'], period)
    spy_prices_adjusted = lookup_prices_vectorized(spy_price_history, pd.to_datetime(df['filing_date_used']))
    spy_diff_pct = count_future_diff_pct(spy_prices_adjusted, period)
    target = count_stock_advantage_over_market(stock_diff_pct, spy_diff_pct)
    target.index = df.index

    ind = compute_indicators(df, period)
    ind = count_past_relative_indicators(ind, period)
    ind['target'] = target.reindex(df.index)


    ind.insert(0, 'fiscalDateEnding', df['fiscalDateEnding'])
    ind.insert(1, 'filing_date_used', df['filing_date_used'])
    ind['fiscalDateEnding'] = pd.to_datetime(ind['fiscalDateEnding'])
    ind['filing_date_used'] = pd.to_datetime(ind['filing_date_used'])
    ind.sort_values('fiscalDateEnding', ascending=True, inplace=True)

    # Remove first 3 years and last year as they are missing Q0/Q_N data or target
    if period == 'quarterly':
        ind = ind[12:-4]
    else:
        ind = ind[3:-1]

    ind = ind.reset_index(drop=True)

    save_data(ind, metadata, output_dir, ticker)
    print(f"[{ticker}] {period} finished successfully")


def save_data(df: pd.DataFrame,
              metadata: dict,
              output_dir: str,
              ticker: str,
    ):
    sector_name = metadata["sector"].lower().replace(" ", "_")

    sector_dir = Path(output_dir) / sector_name
    sector_dir.mkdir(parents=True, exist_ok=True)

    csv_path = sector_dir / f"{ticker}.csv"
    df.to_csv(csv_path, index=False)

    save_metadata(ticker, metadata, f"{output_dir}/metadata.json")

    output_dir_parquet = Path(output_dir + "_parquet")
    output_dir_parquet.mkdir(parents=True, exist_ok=True)

    parquet_path = output_dir_parquet / f"{sector_name}.parquet"

    df_parquet = df.copy()
    df_parquet["ticker"] = ticker
    df_parquet["sector"] = sector_name
    df_parquet["industry"] = metadata["industry"]

    if parquet_path.exists():
        existing = pd.read_parquet(parquet_path)
        existing = existing[existing["ticker"] != ticker]
        combined = pd.concat([existing, df_parquet], ignore_index=True)
    else:
        combined = df_parquet

    combined.to_parquet(parquet_path, index=False)



def process_ticker(ticker: str,
                   metadata: dict,
                   spy_price_history: pd.DataFrame,
                   force: bool = False) -> None:
    sector = metadata["sector"].lower().replace(" ", "_")

    quarterly_csv = os.path.join("data_quarterly", sector, f"{ticker}.csv")
    yearly_csv = os.path.join("data_yearly", sector, f"{ticker}.csv")

    if os.path.exists(quarterly_csv) and os.path.exists(yearly_csv) and not force:
        print(f"{ticker} already parsed, skipping")
        return
        
    price_history = get_price_history(ticker)
    build_reports(ticker,  metadata, 'quarterly', price_history, spy_price_history)
    build_reports(ticker,  metadata, 'yearly', price_history, spy_price_history)

def load_checkpoint_data() -> int:
    if not CHECKPOINT_FILE.exists():
        return 0, []
    
    with open(CHECKPOINT_FILE, 'r') as f:
        data = json.load(f)

    return data["last_completed_index"] + 1, data["skip_tickers"]

def save_checkpoint_data(index: int, tickers_to_skip: list):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({"last_completed_index": index, "skip_tickers": tickers_to_skip}, f)

    if index % 10 == 0:
        print(f"======== {index} company tickers parsed ========")

def is_company_downloaded(ticker):

    base = Path("raw_data") / ticker

    return (
        (base / "income_statement.json").exists() and
        (base / "balance_sheet.json").exists() and
        (base / "cash_flow.json").exists()
    )


In [ ]:
tickers = get_company_tickers()

start_index, tickers_to_skip = load_checkpoint_data()
valid_us_tickers = pd.read_csv(LISTING_PATH)['symbol'].values
spy_price_history = get_price_history('SPY')

for i in range(start_index, len(tickers)-1):
    info = tickers[str(i)]

    ticker = info.get('ticker')
    title = info.get('title')
    
    if ticker not in valid_us_tickers:
        print(f'[{ticker}] not a stock or not traded on major exchange, skipping')
        continue

    if ticker in tickers_to_skip:
        print(f"[{ticker}] previously returned invalid data, skipping")
        continue

    metadata = get_company_metadata(ticker)

    if metadata['sector'] is None:
        print(f'[{ticker}] has unknown sector, skipping')
        tickers_to_skip.append(ticker)
        save_checkpoint_data(i, tickers_to_skip)
        continue

    print(f"Downloading Alpha Vantage data for {ticker}:")

    try:
        download_company(ticker)
        
        if is_company_downloaded(ticker):
            process_ticker(ticker, metadata, spy_price_history)
            save_checkpoint_data(i, tickers_to_skip)
    except RuntimeError as e:
        if str(e) == "ALPHA_VANTAGE_LIMIT":
            print("Alpha Vantage limit reached. Stopping.")
            break
        else:
            print("Error:", ticker, e)
            tickers_to_skip.append(ticker)
            save_checkpoint_data(i, tickers_to_skip)
            os.rmdir(os.path.join("raw_data", ticker))
            time.sleep(1)

In [ ]:
import shutil

# for sector in os.listdir('data_yearly'):
#     if not sector.endswith('.json'):
#         shutil.rmtree(f'data_yearly/{sector}')
#         shutil.rmtree(f'data_quarterly/{sector}')

valid_us_tickers = pd.read_csv(LISTING_PATH)['symbol'].values
spy_price_history = get_price_history('SPY')

for ticker in tqdm(os.listdir("raw_data")):
    if len(os.listdir(f"raw_data/{ticker}")) < 3:
        print(f"Skipping {ticker}, invalid")
        continue

    if ticker not in valid_us_tickers:
        print(f'Ticker {ticker} not a stock or not traded on major exchange, skipping')
        continue
    
    metadata = get_company_metadata(ticker)
    try:
        process_ticker(ticker, metadata, spy_price_history, force=False)
    except Exception as e:
        print(f"[{ticker}] ERROR: {str(e)}, skipping")
        continue


In [ ]:
listed = pd.read_csv('listing_status.csv')
delisted = pd.read_csv('listing_status_delisted.csv')

df = pd.concat([listed, delisted])
df = df[
    (df['assetType'] == 'Stock') &
    (~df['name'].str.contains(r'\bETF\b', case=False, na=False)) &
    (df['exchange'].isin(['NASDAQ', 'NYSE', 'AMEX', 'NYSE MKT']))
]
tickers = df['symbol']
tickers.sort_values()
tickers.to_csv('us_stock_listing.csv', index=False)

In [42]:
frames = []

DIR = "data_quarterly_parquet"
total_missing = 0
total = 0

# for sector in ('financial_services', 'healthcare', 'industrials', 'technology'):
for file in tqdm(os.listdir(DIR)):
    df = pd.read_parquet(os.path.join(DIR, file))
    df = df.drop(columns=['fiscalDateEnding', 'filing_date_used', 'sector', 'industry', 'ticker'])
    df = df.drop(columns=df.filter(like="Q0/").columns)
    print(f"{file} missing: {df.isna().sum().sum()} / {len(df) * len(df.columns)} ({(df.isna().sum().sum() / (len(df) * len(df.columns)) * 100):.2f}%) indicators")
    total_missing += df.isna().sum().sum()
    total += len(df) * len(df.columns)
    frames.append(df)

if frames:
    final_df = pd.concat(frames, ignore_index=True)

print("\nFinal statistics:")
print(f"total missing ratio: {total_missing} / {total} => {round(total_missing / total * 100, 2)}%")
(final_df.isna().mean() * 100).round(2)

  0%|          | 0/11 [00:00<?, ?it/s]

real_estate.parquet missing: 1566 / 494302 (0.32%) indicators
technology.parquet missing: 36560 / 1109495 (3.30%) indicators
energy.parquet missing: 13456 / 479021 (2.81%) indicators
utilities.parquet missing: 1020 / 223846 (0.46%) indicators
basic_materials.parquet missing: 21803 / 512592 (4.25%) indicators
communication_services.parquet missing: 10024 / 350932 (2.86%) indicators
healthcare.parquet missing: 97272 / 1307617 (7.44%) indicators
consumer_cyclical.parquet missing: 14425 / 898039 (1.61%) indicators
consumer_defensive.parquet missing: 7752 / 386450 (2.01%) indicators
industrials.parquet missing: 23775 / 1217937 (1.95%) indicators
financial_services.parquet missing: 27520 / 1470457 (1.87%) indicators

Final statistics:
total missing ratio: 255173 / 8450688 => 3.02%


grossMargin              3.57
operatingMargin          3.54
netProfitMargin          3.52
ebitdaMargin             3.74
fcfMargin                4.53
roe                      0.14
roa                      0.09
equityMultiplier         0.11
capexToRevenue           3.59
capexToOcf               1.16
capexToDepreciation      1.06
growthCapex              1.06
investedCapital          8.36
roic                     8.38
effectiveTaxRate         2.22
ocfToRevenue             4.53
reinvestmentRate         1.09
organicFcf               1.09
organicFcfMargin         4.53
earningsQuality          1.13
grossProfitToAssets      1.07
revenueToAssets          0.67
ownerEarnings            1.06
sloanRatio               3.90
currentRatio             0.41
quickRatio               0.41
debtToEquity             8.36
debtToAssets             8.33
interestCoverage         7.43
netDebtToEbitda          8.46
debtToEbitda             8.41
fcfToEbitda              1.36
eps                      0.06
bookValueP

In [43]:
total_tickers = total_rows = 0
DIR = "data_quarterly_parquet"

for sector in os.listdir(DIR):
    path = os.path.join(DIR, sector)
    df = pd.read_parquet(path)
    total_tickers += len(df['ticker'].unique())
    total_rows += len(df)

    print(f"[{sector[:sector.find('.')]}] Companies: {len(df['ticker'].unique())} Rows: {len(df)}")
print(f"[TOTAL] Companies: {total_tickers} Rows: {total_rows}")

[real_estate] Companies: 218 Rows: 8378
[technology] Companies: 568 Rows: 18805
[energy] Companies: 202 Rows: 8119
[utilities] Companies: 88 Rows: 3794
[basic_materials] Companies: 214 Rows: 8688
[communication_services] Companies: 187 Rows: 5948
[healthcare] Companies: 837 Rows: 22163
[consumer_cyclical] Companies: 423 Rows: 15221
[consumer_defensive] Companies: 169 Rows: 6550
[industrials] Companies: 498 Rows: 20643
[financial_services] Companies: 619 Rows: 24923
[TOTAL] Companies: 4023 Rows: 143232


In [ ]:
# FORM_FALLBACK_DAYS = {'10-Q': 45, '10-K': 75}

ticker = "AAPL"
income, balance, cash = load_raw(ticker)
merged = merge_reports(income, balance, cash, "quarterlyReports")

if any(c not in (None, 'None', 'USD') for c in merged['reportedCurrency'].unique()):
    print(f"ERROR [{ticker}] Company reports not in USD, skipping")
    
price_history = get_price_history(ticker)
merged_final = get_data_after_company_public(merged, price_history)

report_dates = get_report_dates(ticker, 'quarterly')

default_form = '10-Q'
default_fallback = FORM_FALLBACK_DAYS[default_form]

def make_fallback_row():
    return {'filing_date': pd.NaT, 'reportDate': pd.NaT, 'form': default_form}

if report_dates is None:
    print(f"WARNING: [{ticker}] No report dates found, using full default fallback for all rows")
    aligned_dates = pd.DataFrame([make_fallback_row() for _ in range(len(merged_final))])

else:
    report_dates_ts = pd.to_datetime(report_dates['reportDate'])
    aligned_rows = []
    used_positions = set()

    for _, mrow in merged_final.iterrows():
        fiscal_ts = pd.Timestamp(mrow['fiscalDateEnding'])

        # find closest unused reportDate within tolerance
        best_pos = None
        best_delta = None
        for pos_d, report_ts in enumerate(report_dates_ts):
            if pos_d in used_positions:
                continue
            delta = abs(fiscal_ts - report_ts).days
            if delta <= default_fallback:
                if best_delta is None or delta < best_delta:
                    best_pos = pos_d
                    best_delta = delta

        if best_pos is not None:
            used_positions.add(best_pos)
            aligned_rows.append(report_dates.iloc[best_pos].to_dict())
        else:
            aligned_rows.append(make_fallback_row())

    aligned_dates = pd.DataFrame(aligned_rows).reset_index(drop=True)
    
    n_fallback = sum(1 for r in aligned_rows if pd.isna(r['filing_date']))
    if n_fallback > 0:
        print(f"WARNING: [{ticker}] {n_fallback} rows using default fallback (no matching reportDate)")
# else:
    # anchor_merged_idx = None
    # anchor_dates_idx = None

    # for pos_m, (mi, mrow) in enumerate(merged_final.iterrows()):
    #     fiscal_ts = pd.Timestamp(mrow['fiscalDateEnding'])
    #     for pos_d, (di, drow) in enumerate(report_dates.iterrows()):
    #         if abs(fiscal_ts - pd.Timestamp(drow['reportDate'])).days <= default_fallback:
    #             anchor_merged_idx = pos_m
    #             anchor_dates_idx = pos_d
    #             break
    #     if anchor_merged_idx is not None:
    #         break
    
    # if anchor_merged_idx is None:
    #     print(f"WARNING: [{ticker}] No matching reportDate found for any fiscalDateEnding, using full default fallback")
    #     aligned_dates = pd.DataFrame([make_fallback_row() for _ in range(len(merged_final))])
    # else:
    #     n_fallback = anchor_merged_idx
    #     real_dates = report_dates.iloc[anchor_dates_idx:].reset_index(drop=True)
    #     n_real_available = len(real_dates)
    #     n_real_needed = len(merged_final) - n_fallback

    #     if n_real_available < n_real_needed:
    #         print(f"WARNING: [{ticker}] report_dates runs out after anchor, padding {n_real_needed - n_real_available} trailing rows with fallback")
    #         trailing = pd.DataFrame([make_fallback_row() for _ in range(n_real_needed - n_real_available)])
    #         real_dates = pd.concat([real_dates, trailing], ignore_index=True)
    #     else:
    #         real_dates = real_dates.iloc[:n_real_needed].reset_index(drop=True)

    #     fallback_rows = pd.DataFrame([make_fallback_row() for _ in range(n_fallback)])
    #     aligned_dates = pd.concat([fallback_rows, real_dates], ignore_index=True)


df = pd.concat([
    merged_final.reset_index(drop=True),
    aligned_dates.reset_index(drop=True)
], axis=1)

fallback_days = df['form'].map(FORM_FALLBACK_DAYS).fillna(45)
delta = (df['fiscalDateEnding'] - pd.to_datetime(df['filing_date'])).dt.days.abs()

has_filing = df['filing_date'].notna() & delta.notna() & (delta <= fallback_days)

df['filing_date_used'] = np.where(
    has_filing,
    df['filing_date'].astype(str),
    (df['fiscalDateEnding'] + pd.to_timedelta(fallback_days, unit='D')).astype(str)
)

# df['stock_price'] = lookup_prices_vectorized(price_history, pd.to_datetime(df['filing_date_used']))
df[['fiscalDateEnding', 'reportDate', 'filing_date_used', 'form']].tail()
# df = apply_accounting_logic_av(df)
# ind = compute_indicators(df, 'quarterly')
# # ind = count_past_relative_indicators(ind, 'quarterly')

# ind.insert(0, 'fiscalDateEnding', df['fiscalDateEnding'])
# ind.insert(1, 'filing_date_used', df['filing_date_used'])
# ind['fiscalDateEnding'] = pd.to_datetime(ind['fiscalDateEnding'])
# ind['filing_date_used'] = pd.to_datetime(ind['filing_date_used'])
# ind.sort_values('fiscalDateEnding', ascending=False, inplace=True)
# ind = ind.reset_index(drop=True)
